In [1]:
# InGame Score Prediction Model Notebook 1: Live Data Retrieval & Raw Data Acquisition

In [ ]:
# Header For Imports

import sys
import os

# Determine the project root (assuming structure: score-genius/ with backend/ and notebooks/ as siblings)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Notebook 1: Live Data Retrieval
import pandas as pd
import pytz
from datetime import datetime
import traceback
from caching.supabase_client import supabase

from backend.models.features import NBAFeatureGenerator

In [3]:
# Cell 1: Imports, Live Game Data Retrieval in Pacific Time

import pandas as pd
import pytz
from datetime import datetime
import traceback
from caching.supabase_client import supabase

def fetch_live_games_in_pacific_time():
    """
    Fetch actively live games from 'nba_live_game_stats' using the stored 'status'
    field. A game is considered live if its status is one of:
      {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    """
    live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    
    try:
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No data found in 'nba_live_game_stats' table.")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        
        # Ensure 'game_date' exists
        if "game_date" not in df.columns:
            print("Column 'game_date' not found!")
            return pd.DataFrame()
        
        # Convert game_date (stored as fake UTC) to datetime and then to Pacific Time
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
        pacific_tz = pytz.timezone("America/Los_Angeles")
        df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
        df["date_only_pt"] = df["game_date_pt"].dt.date
        
        # Filter for games scheduled for today in Pacific Time
        now_pt = datetime.now(pacific_tz)
        today_pt = now_pt.date()
        df_today = df[df["date_only_pt"] == today_pt].copy()
        if df_today.empty:
            print("No games match today's date in Pacific Time.")
            return pd.DataFrame()
        
        # Make sure the status field is available and in uppercase
        if "status" not in df_today.columns:
            print("Column 'status' not found!")
            return pd.DataFrame()
        df_today["status"] = df_today["status"].astype(str).str.upper()
        
        # Filter only active games (using the stored status)
        active_games = df_today[df_today["status"].isin(live_statuses)].copy()
        
        print(f"ACTIVELY LIVE GAMES: {len(active_games)}")
        if not active_games.empty:
            for _, row in active_games.iterrows():
                print(
                    f"{row.get('home_team', 'Unknown')} vs {row.get('away_team', 'Unknown')}: "
                    f"{row.get('home_score', 'N/A')}-{row.get('away_score', 'N/A')}, "
                    f"Status: {row.get('status')}"
                )
        else:
            print("No active games found!")
        
        return active_games
        
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Test the function
live_games = fetch_live_games_in_pacific_time()
if not live_games.empty:
    print("Successfully retrieved active games (Pacific Time).")
else:
    print("No active game data available.")

ACTIVELY LIVE GAMES: 3
Los Angeles Lakers vs San Antonio Spurs: 94-81, Status: Q4
Golden State Warriors vs Denver Nuggets: 92-97, Status: Q4
Sacramento Kings vs Memphis Grizzlies: 132-118, Status: Q4
Successfully retrieved active games (Pacific Time).


In [ ]:
# Cell 3: Get Raw Live Game Data from Supabase with Error Handling

import pandas as pd
import time
import pytz
from datetime import datetime
from caching.supabase_client import supabase
import traceback

def convert_time_to_minutes(time_str: str) -> float:
    if pd.isna(time_str) or ":" not in str(time_str):
        return None
    try:
        minutes, seconds = str(time_str).split(":")
        return float(minutes) + float(seconds) / 60.0
    except Exception as e:
        print(f"Error converting time: {e}")
        return None
    
def add_basketball_metrics_to_features(feature_sets):
    """
    Enhance the existing feature sets with basketball analytics metrics and team form.
    
    Args:
        feature_sets: Dictionary with feature lists for each quarter
    Returns:
        Updated feature sets with basketball analytics metrics
    """
    # These are the basketball analytics metrics to potentially add
    shooting_metrics = ['home_fg_pct', 'away_fg_pct', 'fg_pct_diff']
    free_throw_metrics = ['home_ft_pct', 'away_ft_pct', 'ft_pct_diff', 'home_ft_rate', 'away_ft_rate']
    advanced_metrics = ['home_possessions', 'away_possessions', 'game_pace', 
                       'home_off_efficiency', 'away_off_efficiency', 'efficiency_diff',
                       'momentum_efficiency', 'pace_adj_diff']
    form_metrics = ['home_form_score', 'form_score_diff', 'home_streak', 'streak_diff', 'total_momentum']
    
    # Enhanced feature sets with basketball metrics
    enhanced_sets = {}
    
    for quarter, features in feature_sets.items():
        # Start with existing features
        enhanced_features = features.copy()
        
        # Add appropriate metrics based on quarter
        if quarter == 'q1':
            # For pre-game, team form is especially important
            enhanced_features.extend([
                'fg_pct_diff',
                'ft_pct_diff',
                'efficiency_diff',
                'home_form_score',
                'streak_diff'  # Form is key for pre-game prediction
            ])
        elif quarter == 'q2':
            # In Q2, current form still matters but in-game stats start to take over
            enhanced_features.extend([
                'fg_pct_diff',
                'ft_pct_diff',
                'game_pace',
                'efficiency_diff',
                'home_form_score',
                'total_momentum'  # Combined momentum and form
            ])
        elif quarter == 'q3':
            # In Q3, form becomes less important as game stats take over
            enhanced_features.extend([
                'fg_pct_diff',
                'ft_pct_diff',
                'game_pace',
                'efficiency_diff',
                'momentum_efficiency',
                'total_momentum'
            ])
        elif quarter == 'q4':
            # In Q4, use all available metrics
            enhanced_features.extend([
                'fg_pct_diff',
                'ft_pct_diff',
                'game_pace',
                'efficiency_diff',
                'momentum_efficiency',
                'pace_adj_diff',
                'total_momentum'  # Still include form in late game scenarios
            ])
        
        enhanced_sets[quarter] = enhanced_features
    
    return enhanced_sets

# Example usage:
# quarter_feature_sets = get_quarter_feature_sets()
# enhanced_feature_sets = add_basketball_metrics_to_features(quarter_feature_sets)

def get_active_live_games():
    max_retries = 3
    retry_delay = 2  # seconds
    
    for attempt in range(max_retries):
        try:
            print(f"Attempting to fetch live game data (attempt {attempt+1}/{max_retries})...")
            response = supabase.table("nba_live_game_stats").select("*").execute()
            raw_data = response.data
            
            if not raw_data:
                print("No live game data available.")
                return pd.DataFrame()
                
            # Convert to DataFrame
            raw_df = pd.DataFrame(raw_data)
            print(f"Retrieved {len(raw_df)} games from nba_live_game_stats")
            
            # Process minutes if present
            if 'minutes' in raw_df.columns:
                raw_df['minutes_numeric'] = raw_df['minutes'].apply(convert_time_to_minutes)
                raw_df.drop(columns=['minutes'], inplace=True)
            
            # Convert game_date to datetime and filter for today's games in Pacific Time
            if "game_date" in raw_df.columns:
                raw_df["game_date"] = pd.to_datetime(raw_df["game_date"], errors="coerce", utc=True)
                pacific_tz = pytz.timezone("America/Los_Angeles")
                raw_df["game_date_pt"] = raw_df["game_date"].dt.tz_convert(pacific_tz)
                raw_df["date_only_pt"] = raw_df["game_date_pt"].dt.date
                
                now_pt = datetime.now(pacific_tz)
                today_pt = now_pt.date()
                raw_df = raw_df[raw_df["date_only_pt"] == today_pt].copy()
                print(f"Found {len(raw_df)} games scheduled for today")
            
            # Ensure the 'status' column exists and convert to uppercase
            if "status" not in raw_df.columns:
                print("Column 'status' not found!")
                return pd.DataFrame()
            raw_df["status"] = raw_df["status"].astype(str).str.upper()
            
            # Mark active games based solely on the status field
            live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
            raw_df['is_active_now'] = raw_df["status"].isin(live_statuses)
            
            active_df = raw_df[raw_df['is_active_now']].copy()
            print(f"Found {len(active_df)} actively live games")
            
            if active_df.empty:
                print("No actively live games found.")
                return pd.DataFrame()
            
            # Optionally, check for missing values in critical columns
            critical_cols = ['home_q1', 'home_q2', 'home_q3', 'home_q4', 
                             'away_q1', 'away_q2', 'away_q3', 'away_q4',
                             'home_score', 'away_score', 'current_quarter']
            missing_data = {}
            for col in critical_cols:
                if col in active_df.columns:
                    missing_count = active_df[col].isna().sum()
                    if missing_count > 0:
                        missing_data[col] = missing_count
            if missing_data:
                print("\nWARNING: Missing values detected in critical columns:")
                for col, count in missing_data.items():
                    print(f"  • {col}: {count} missing values ({count/len(active_df)*100:.1f}%)")
            
            return active_df
            
        except Exception as e:
            print(f"Connection error: {e}")
            traceback.print_exc()
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 2
            else:
                print("Maximum retries reached. Please check your network connection and Supabase configuration.")
                return pd.DataFrame()

# Get actively live games
raw_df = get_active_live_games()

# Display the results
if not raw_df.empty:
    print("\nLatest Active Game Data:")
    display(raw_df)
else:
    print("\nNo actively live game data available for analysis.")


Attempting to fetch live game data (attempt 1/3)...
Retrieved 10 games from nba_live_game_stats
Found 10 games scheduled for today
Found 3 actively live games

Latest Active Game Data:


,id,game_id,home_team,away_team,home_score,away_score,home_q1,home_q2,home_q3,home_q4,...,home_fg_attempted,away_fg_made,away_fg_attempted,home_ft_made,home_ft_attempted,away_ft_made,away_ft_attempted,game_date_pt,date_only_pt,is_active_now
0,390,414411,Los Angeles Lakers,San Antonio Spurs,94,81,32,33,29,0,...,66,28,64,15,20,9,10,2025-03-17 12:30:00-07:00,2025-03-17,True
6,386,414863,Golden State Warriors,Denver Nuggets,92,97,22,28,27,15,...,74,35,76,8,19,12,14,2025-03-17 12:00:00-07:00,2025-03-17,True
8,388,414865,Sacramento Kings,Memphis Grizzlies,132,118,37,28,33,34,...,85,43,86,18,28,19,19,2025-03-17 12:00:00-07:00,2025-03-17,True
